In [1]:
# Cell 1 - Project setup

import os
import sys
import json
import re
import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import skrf as rf

PROJECT_ROOT = "/home/tekb/Master_Thesis_KB/CodesAndData"

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Device:", device)

PROJECT_ROOT: /home/tekb/Master_Thesis_KB/CodesAndData
Device: cuda


In [2]:
# Cell 2 - EM validation configuration for random-start inverse design

EM_CFG = {
    # Input folder containing EM .s4p files
    "em_input_dir": os.path.join(
        PROJECT_ROOT,
        "Data",
        "random_inverse_design",
    ),

    # Output folder for validation results
    "em_results_dir": os.path.join(
        PROJECT_ROOT,
        "Results",
        "EM_Validation_Random",
    ),

    # Impedance cache used as reference database
    "impedance_cache": os.path.join(
        PROJECT_ROOT,
        "Data",
        "stage2_impedance_cache_ln_phase_full20k.pt",
    ),

    # Expected file pattern:
    # target_100005_start_116230_geometry_impedance_loss_out.s4p
    "filename_regex": r"target_(\d+)_start_(\d+)_geometry_impedance_loss_out\.s4p",

    # Plot settings
    "save_plots": True,
    "dpi": 300,
}

os.makedirs(EM_CFG["em_results_dir"], exist_ok=True)

print("EM input folder:", EM_CFG["em_input_dir"])
print("EM results folder:", EM_CFG["em_results_dir"])

EM input folder: /home/tekb/Master_Thesis_KB/CodesAndData/Data/random_inverse_design
EM results folder: /home/tekb/Master_Thesis_KB/CodesAndData/Results/EM_Validation_Random


In [3]:
# Cell 3 - Load impedance cache

cache_path = EM_CFG["impedance_cache"]

if not os.path.exists(cache_path):
    raise FileNotFoundError(f"Impedance cache not found:\n{cache_path}")

imp_cache = torch.load(cache_path, map_location="cpu")

cache_sids = imp_cache["simu_ids"].cpu().numpy().astype(int)
sid_to_cache_pos = {int(s): i for i, s in enumerate(cache_sids)}

freq_hz = np.asarray(imp_cache["freq_hz"], dtype=np.float64)
freq_mhz = freq_hz / 1e6

impedance_data = imp_cache["impedance"]

print("Loaded impedance cache:")
print(cache_path)
print("Impedance shape:", impedance_data.shape)
print("Frequency points:", len(freq_hz))
print("Number of SIDs:", len(cache_sids))

Loaded impedance cache:
/home/tekb/Master_Thesis_KB/CodesAndData/Data/stage2_impedance_cache_ln_phase_full20k.pt
Impedance shape: torch.Size([20000, 2, 334])
Frequency points: 334
Number of SIDs: 20000


/tmp/ipykernel_286151/2131589225.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  imp_cache = torch.load(cache_path, map_location="cpu")


In [4]:
# Cell 4 - Helper functions

def get_true_impedance_by_sid(sid):
    """
    Get database/reference impedance for one simulation ID.

    Returns:
        mag_ohm   : |Z11| in Ohm, shape [F]
        phase_rad : phase in radians, shape [F]
        y         : raw ln_phase data, shape [2, F]
    """
    sid = int(sid)

    if sid not in sid_to_cache_pos:
        raise KeyError(f"SID {sid} not found in impedance cache.")

    pos = sid_to_cache_pos[sid]

    y = impedance_data[pos].cpu().numpy()  # [2, F]
    mag_ohm = np.exp(y[0])
    phase_rad = y[1]

    return mag_ohm, phase_rad, y


def load_z11_from_s4p(s4p_path):
    """
    Load Z11 from an EM .s4p file using scikit-rf.

    Returns:
        freq_hz   : frequency array [F]
        z11       : complex Z11 [F]
        mag_ohm   : |Z11| [F]
        phase_rad : phase(Z11) [F]
    """
    ntwk = rf.Network(s4p_path)

    freq_hz_em = ntwk.f
    z = ntwk.z
    z11 = z[:, 0, 0]

    mag_ohm = np.abs(z11)
    phase_rad = np.angle(z11)

    return freq_hz_em, z11, mag_ohm, phase_rad


def mae_ohm(a, b):
    return float(np.mean(np.abs(a - b)))


def mae_db(a, b):
    a_db = 20.0 * np.log10(a + 1e-12)
    b_db = 20.0 * np.log10(b + 1e-12)
    return float(np.mean(np.abs(a_db - b_db)))


def improvement_pct(baseline_error, optimized_error):
    return float(100.0 * (baseline_error - optimized_error) / baseline_error)


def parse_em_filename(s4p_path, regex):
    """
    Parse target_sid and start_sid from filename.
    Expected:
        target_100005_start_116230_geometry_impedance_loss_out.s4p
    """
    fname = os.path.basename(s4p_path)
    match = re.match(regex, fname)

    if match is None:
        raise ValueError(f"Filename does not match expected pattern:\n{fname}")

    target_sid = int(match.group(1))
    start_sid = int(match.group(2))

    return target_sid, start_sid

In [5]:
# Cell 5 - Discover random-start EM .s4p files

s4p_files = sorted(glob.glob(os.path.join(EM_CFG["em_input_dir"], "*.s4p")))

if len(s4p_files) == 0:
    raise FileNotFoundError(
        f"No .s4p files found in:\n{EM_CFG['em_input_dir']}"
    )

EM_VALIDATION_FILES = []

for s4p_path in s4p_files:
    target_sid, start_sid = parse_em_filename(
        s4p_path=s4p_path,
        regex=EM_CFG["filename_regex"],
    )

    EM_VALIDATION_FILES.append({
        "design_type": "random",
        "target_sid": target_sid,
        "start_sid": start_sid,
        "s4p_path": s4p_path,
        "s4p_file": os.path.basename(s4p_path),
    })

print("Found random-start EM validation files:", len(EM_VALIDATION_FILES))

for item in EM_VALIDATION_FILES:
    print(
        f"Target {item['target_sid']} | "
        f"Random start {item['start_sid']} | "
        f"{item['s4p_file']}"
    )

Found random-start EM validation files: 5
Target 100005 | Random start 116230 | target_100005_start_116230_geometry_impedance_loss_out.s4p
Target 100200 | Random start 116230 | target_100200_start_116230_geometry_impedance_loss_out.s4p
Target 109000 | Random start 116230 | target_109000_start_116230_geometry_impedance_loss_out.s4p
Target 119000 | Random start 116229 | target_119000_start_116229_geometry_impedance_loss_out.s4p
Target 119090 | Random start 116229 | target_119090_start_116229_geometry_impedance_loss_out.s4p


In [6]:
# Cell 6 - Validate one random-start EM result

def validate_one_em_result(item, save_plots=True):
    """
    Validate one EM-simulated random-start inverse-designed geometry.

    Compares:
        target true impedance
        random-start true impedance
        EM simulated optimized geometry impedance
    """
    target_sid = int(item["target_sid"])
    start_sid = int(item["start_sid"])
    s4p_path = item["s4p_path"]

    target_mag, target_phase, _ = get_true_impedance_by_sid(target_sid)
    start_mag, start_phase, _ = get_true_impedance_by_sid(start_sid)

    em_freq_hz, em_z11, em_mag, em_phase = load_z11_from_s4p(s4p_path)
    em_freq_mhz = em_freq_hz / 1e6

    same_len = len(em_freq_hz) == len(freq_hz)
    same_grid = same_len and np.allclose(
        em_freq_hz,
        freq_hz,
        rtol=1e-6,
        atol=1e-3,
    )

    if not same_grid:
        print(f"Warning: frequency grid differs for target {target_sid}, start {start_sid}.")
        print("Metrics assume same frequency grid. Consider interpolation if needed.")

    random_true_mae_ohm = mae_ohm(target_mag, start_mag)
    random_true_mae_db = mae_db(target_mag, start_mag)

    em_optimized_mae_ohm = mae_ohm(target_mag, em_mag)
    em_optimized_mae_db = mae_db(target_mag, em_mag)

    improvement_ohm = improvement_pct(random_true_mae_ohm, em_optimized_mae_ohm)
    improvement_db = improvement_pct(random_true_mae_db, em_optimized_mae_db)

    result = {
        "design_type": "random",
        "target_sid": target_sid,
        "random_start_sid": start_sid,
        "s4p_file": item["s4p_file"],
        "s4p_path": s4p_path,

        "random_start_true_mae_ohm": float(random_true_mae_ohm),
        "random_start_true_mae_db": float(random_true_mae_db),

        "em_optimized_mae_ohm": float(em_optimized_mae_ohm),
        "em_optimized_mae_db": float(em_optimized_mae_db),

        "em_improvement_over_random_start_ohm_pct": float(improvement_ohm),
        "em_improvement_over_random_start_db_pct": float(improvement_db),

        "same_frequency_grid": bool(same_grid),
        "n_freq_database": int(len(freq_hz)),
        "n_freq_em": int(len(em_freq_hz)),
    }

    run_dir = os.path.join(
        EM_CFG["em_results_dir"],
        f"target_{target_sid}_start_{start_sid}"
    )

    os.makedirs(run_dir, exist_ok=True)

    metrics_path = os.path.join(run_dir, "metrics_em_validation.json")

    with open(metrics_path, "w") as f:
        json.dump(result, f, indent=2)

    if save_plots:
        # Magnitude plot
        mag_path = os.path.join(run_dir, "em_validation_magnitude.png")

        plt.figure(figsize=(9, 6))
        plt.loglog(
            freq_mhz,
            target_mag,
            "k-",
            linewidth=2.2,
            label=f"Target true SID={target_sid}",
        )
        plt.loglog(
            freq_mhz,
            start_mag,
            "--",
            linewidth=1.7,
            label=f"Random start true SID={start_sid}",
        )
        plt.loglog(
            em_freq_mhz,
            em_mag,
            "-.",
            linewidth=2.0,
            label="EM simulated optimized geometry",
        )

        plt.xlabel("Frequency (MHz)")
        plt.ylabel("|Z11| (Ohm)")
        plt.title(
            f"EM Validation of Random-Start Inverse-Designed Geometry\n"
            f"Target SID={target_sid}, Start SID={start_sid}"
        )
        plt.grid(True, which="both", alpha=0.3)
        plt.legend()
        plt.tight_layout()
        plt.savefig(mag_path, dpi=EM_CFG["dpi"], bbox_inches="tight")
        plt.close()

        # Phase plot
        phase_path = os.path.join(run_dir, "em_validation_phase.png")

        plt.figure(figsize=(9, 5))
        plt.semilogx(
            freq_mhz,
            np.rad2deg(target_phase),
            "k-",
            linewidth=2.2,
            label=f"Target true SID={target_sid}",
        )
        plt.semilogx(
            freq_mhz,
            np.rad2deg(start_phase),
            "--",
            linewidth=1.7,
            label=f"Random start true SID={start_sid}",
        )
        plt.semilogx(
            em_freq_mhz,
            np.rad2deg(em_phase),
            "-.",
            linewidth=2.0,
            label="EM simulated optimized geometry",
        )

        plt.xlabel("Frequency (MHz)")
        plt.ylabel("Phase (degree)")
        plt.ylim(-185, 185)
        plt.yticks([-180, -90, 0, 90, 180])
        plt.title(
            f"EM Validation Phase Comparison - Random Start\n"
            f"Target SID={target_sid}, Start SID={start_sid}"
        )
        plt.grid(True, which="both", alpha=0.3)
        plt.legend()
        plt.tight_layout()
        plt.savefig(phase_path, dpi=EM_CFG["dpi"], bbox_inches="tight")
        plt.close()

    print(
        f"Done target {target_sid}, random start {start_sid}: "
        f"random={random_true_mae_db:.3f} dB, "
        f"EM={em_optimized_mae_db:.3f} dB, "
        f"improvement={improvement_db:.2f}%"
    )

    return result

In [7]:
# Cell 7 - Run EM validation for all random-start files

all_em_results = []

for item in EM_VALIDATION_FILES:
    result = validate_one_em_result(
        item=item,
        save_plots=EM_CFG["save_plots"],
    )
    all_em_results.append(result)

em_results_df = pd.DataFrame(all_em_results)

display(em_results_df)

Done target 100005, random start 116230: random=4.632 dB, EM=3.467 dB, improvement=25.16%
Done target 100200, random start 116230: random=6.857 dB, EM=4.421 dB, improvement=35.53%
Done target 109000, random start 116230: random=3.379 dB, EM=1.999 dB, improvement=40.85%
Done target 119000, random start 116229: random=11.539 dB, EM=3.829 dB, improvement=66.82%
Done target 119090, random start 116229: random=16.584 dB, EM=4.482 dB, improvement=72.97%


,design_type,target_sid,random_start_sid,s4p_file,s4p_path,random_start_true_mae_ohm,random_start_true_mae_db,em_optimized_mae_ohm,em_optimized_mae_db,em_improvement_over_random_start_ohm_pct,em_improvement_over_random_start_db_pct,same_frequency_grid,n_freq_database,n_freq_em
0,random,100005,116230,target_100005_start_116230_geometry_impedance_...,/home/tekb/Master_Thesis_KB/CodesAndData/Data/...,0.923919,4.632488,0.609098,3.467099,34.074532,25.156861,True,334,334
1,random,100200,116230,target_100200_start_116230_geometry_impedance_...,/home/tekb/Master_Thesis_KB/CodesAndData/Data/...,2.767059,6.857282,2.018534,4.421015,27.051295,35.528175,True,334,334
2,random,109000,116230,target_109000_start_116230_geometry_impedance_...,/home/tekb/Master_Thesis_KB/CodesAndData/Data/...,0.810095,3.379093,0.479161,1.998725,40.851319,40.850271,True,334,334
3,random,119000,116229,target_119000_start_116229_geometry_impedance_...,/home/tekb/Master_Thesis_KB/CodesAndData/Data/...,5.413420,11.539412,0.727343,3.828692,86.564079,66.820738,True,334,334
4,random,119090,116229,target_119090_start_116229_geometry_impedance_...,/home/tekb/Master_Thesis_KB/CodesAndData/Data/...,6.257887,16.584082,0.377166,4.482455,93.972943,72.971339,True,334,334


In [8]:
# Cell 8 - Save random-start EM validation summary

summary_csv_path = os.path.join(
    EM_CFG["em_results_dir"],
    "em_validation_summary_random.csv"
)

em_results_df.to_csv(summary_csv_path, index=False)

print("Saved random-start EM validation summary:")
print(summary_csv_path)

display(
    em_results_df[
        [
            "target_sid",
            "random_start_sid",
            "random_start_true_mae_ohm",
            "em_optimized_mae_ohm",
            "em_improvement_over_random_start_ohm_pct",
            "random_start_true_mae_db",
            "em_optimized_mae_db",
            "em_improvement_over_random_start_db_pct",
            "same_frequency_grid",
        ]
    ].sort_values("em_improvement_over_random_start_db_pct", ascending=False)
)

Saved random-start EM validation summary:
/home/tekb/Master_Thesis_KB/CodesAndData/Results/EM_Validation_Random/em_validation_summary_random.csv


,target_sid,random_start_sid,random_start_true_mae_ohm,em_optimized_mae_ohm,em_improvement_over_random_start_ohm_pct,random_start_true_mae_db,em_optimized_mae_db,em_improvement_over_random_start_db_pct,same_frequency_grid
4,119090,116229,6.257887,0.377166,93.972943,16.584082,4.482455,72.971339,True
3,119000,116229,5.413420,0.727343,86.564079,11.539412,3.828692,66.820738,True
2,109000,116230,0.810095,0.479161,40.851319,3.379093,1.998725,40.850271,True
1,100200,116230,2.767059,2.018534,27.051295,6.857282,4.421015,35.528175,True
0,100005,116230,0.923919,0.609098,34.074532,4.632488,3.467099,25.156861,True


In [9]:
# Cell 9 - Aggregate random-start EM validation statistics

n_total = len(em_results_df)

n_success_ohm = int((em_results_df["em_improvement_over_random_start_ohm_pct"] > 0).sum())
n_success_db = int((em_results_df["em_improvement_over_random_start_db_pct"] > 0).sum())

avg_ohm_all = float(em_results_df["em_improvement_over_random_start_ohm_pct"].mean())
avg_db_all = float(em_results_df["em_improvement_over_random_start_db_pct"].mean())

positive_df = em_results_df[em_results_df["em_improvement_over_random_start_db_pct"] > 0]

avg_ohm_positive = float(positive_df["em_improvement_over_random_start_ohm_pct"].mean())
avg_db_positive = float(positive_df["em_improvement_over_random_start_db_pct"].mean())

print("Random-start EM validation aggregate statistics")
print("-----------------------------------------------")
print(f"Total cases                       : {n_total}")
print(f"Successful cases in Ohm MAE        : {n_success_ohm}/{n_total}")
print(f"Successful cases in dB MAE         : {n_success_db}/{n_total}")
print(f"Average Ohm improvement all cases  : {avg_ohm_all:.2f}%")
print(f"Average dB improvement all cases   : {avg_db_all:.2f}%")
print(f"Average Ohm improvement successes  : {avg_ohm_positive:.2f}%")
print(f"Average dB improvement successes   : {avg_db_positive:.2f}%")

Random-start EM validation aggregate statistics
-----------------------------------------------
Total cases                       : 5
Successful cases in Ohm MAE        : 5/5
Successful cases in dB MAE         : 5/5
Average Ohm improvement all cases  : 56.50%
Average dB improvement all cases   : 48.27%
Average Ohm improvement successes  : 56.50%
Average dB improvement successes   : 48.27%
